# 🚀 放物運動シミュレーター / Projectile Motion Simulator

空気抵抗がある場合とない場合のボールの放物運動を比較・学習できるインタラクティブシミュレーターです。

---

## 📚 物理学の背景

### 空気抵抗なし（理想的な放物運動）
$$x(t) = v_0 \cos(\theta) \cdot t$$
$$y(t) = h_0 + v_0 \sin(\theta) \cdot t - \frac{1}{2}gt^2$$

### 空気抵抗あり（二次抵抗モデル）
$$F_{drag} = \frac{1}{2} \rho C_d A v^2$$

- $\rho$ = 空気密度 (1.225 kg/m³)
- $C_d$ = 抗力係数 (球体 ≈ 0.47)
- $A$ = 断面積 ($\pi r^2$)
- $v$ = 速度の大きさ

---

## 使い方
1. **セルを上から順番に実行** してください
2. スライダーでパラメーターを調整すると **リアルタイムで軌跡が更新** されます
3. 結果の数値は軌跡グラフの下に表示されます

---

## 📝 グラフの英語表記と日本語対訳

グラフ内のラベルは文字化け防止のため英語表記になっています。以下が対訳表です：

| 英語 (グラフ表記) | 日本語 |
|------------------|--------|
| Distance [m] | 距離 [m] |
| Height [m] | 高さ [m] |
| Range [m] | 飛距離 [m] |
| Launch Angle [deg] | 発射角度 [度] |
| No Air Resistance | 空気抵抗なし |
| With Air Resistance | 空気抵抗あり |
| Ideal (No Drag) | 理想軌跡（抵抗なし） |
| Launch Point | 発射点 |
| Optimal Angle | 最適角度 |

## 1️⃣ ライブラリのインポート

In [ ]:
# 必要なライブラリのインポート
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, Layout, HBox, VBox

# プロットスタイルの設定
plt.style.use('seaborn-v0_8-whitegrid')

print("✅ ライブラリの読み込み完了！")

## 2️⃣ 物理エンジンの定義

In [ ]:
class PhysicsEngine:
    """
    放物運動の物理演算エンジン
    
    - 空気抵抗なし: 解析解（厳密解）
    - 空気抵抗あり: Runge-Kutta法による数値解法
    """
    
    # 物理定数
    GRAVITY = 9.81       # 重力加速度 [m/s²]
    AIR_DENSITY = 1.225  # 空気密度 [kg/m³]（海面、15℃）
    
    @staticmethod
    def no_air_resistance(v0, angle_deg, height=0, dt=0.01):
        """
        空気抵抗なしの放物運動（解析解）
        
        Parameters:
        -----------
        v0 : float - 初速度 [m/s]
        angle_deg : float - 発射角度 [度]
        height : float - 初期高さ [m]
        dt : float - 時間刻み [s]
        
        Returns:
        --------
        dict: 軌跡データと統計情報
        """
        angle_rad = np.radians(angle_deg)
        v0x = v0 * np.cos(angle_rad)
        v0y = v0 * np.sin(angle_rad)
        g = PhysicsEngine.GRAVITY
        
        # 飛行時間の計算
        discriminant = v0y**2 + 2 * g * height
        if discriminant < 0:
            return {'x': np.array([0]), 'y': np.array([height]),
                    'time': 0, 'max_height': height, 'range': 0}
        
        t_flight = (v0y + np.sqrt(discriminant)) / g
        t = np.arange(0, t_flight + dt, dt)
        
        # 軌跡の計算
        x = v0x * t
        y = height + v0y * t - 0.5 * g * t**2
        
        # y >= 0 の部分のみ
        valid = y >= 0
        x, y = x[valid], y[valid]
        
        # 最大高さ
        max_height = height + v0y**2 / (2 * g) if v0y > 0 else height
        
        return {
            'x': x, 'y': y,
            'time': t_flight,
            'max_height': max_height,
            'range': x[-1] if len(x) > 0 else 0
        }
    
    @staticmethod
    def with_air_resistance(v0, angle_deg, mass, radius,
                            drag_coefficient=0.47, height=0, dt=0.001):
        """
        空気抵抗ありの放物運動（Runge-Kutta 4次法）
        
        空気抵抗モデル: F_drag = 0.5 * ρ * Cd * A * v²
        
        Parameters:
        -----------
        v0 : float - 初速度 [m/s]
        angle_deg : float - 発射角度 [度]
        mass : float - ボールの質量 [kg]
        radius : float - ボールの半径 [m]
        drag_coefficient : float - 抗力係数 Cd
        height : float - 初期高さ [m]
        dt : float - 時間刻み [s]
        
        Returns:
        --------
        dict: 軌跡データと統計情報
        """
        angle_rad = np.radians(angle_deg)
        g = PhysicsEngine.GRAVITY
        rho = PhysicsEngine.AIR_DENSITY
        
        # 断面積と抵抗定数
        A = np.pi * radius**2
        k = 0.5 * rho * drag_coefficient * A / mass
        
        # 初期条件
        x, y = 0.0, height
        vx = v0 * np.cos(angle_rad)
        vy = v0 * np.sin(angle_rad)
        
        x_list, y_list = [x], [y]
        t = 0
        max_height = height
        
        def acceleration(vx, vy):
            """加速度を計算"""
            v = np.sqrt(vx**2 + vy**2)
            if v > 0:
                return -k * v * vx, -g - k * v * vy
            return 0, -g
        
        # Runge-Kutta 4次法
        while y >= 0 and t < 1000:
            # k1
            ax1, ay1 = acceleration(vx, vy)
            
            # k2
            vx2, vy2 = vx + ax1*dt/2, vy + ay1*dt/2
            ax2, ay2 = acceleration(vx2, vy2)
            
            # k3
            vx3, vy3 = vx + ax2*dt/2, vy + ay2*dt/2
            ax3, ay3 = acceleration(vx3, vy3)
            
            # k4
            vx4, vy4 = vx + ax3*dt, vy + ay3*dt
            ax4, ay4 = acceleration(vx4, vy4)
            
            # 更新
            x += (vx + 2*vx2 + 2*vx3 + vx4) / 6 * dt
            y += (vy + 2*vy2 + 2*vy3 + vy4) / 6 * dt
            vx += (ax1 + 2*ax2 + 2*ax3 + ax4) / 6 * dt
            vy += (ay1 + 2*ay2 + 2*ay3 + ay4) / 6 * dt
            t += dt
            
            if y > max_height:
                max_height = y
            
            if y >= 0:
                x_list.append(x)
                y_list.append(y)
        
        x_arr, y_arr = np.array(x_list), np.array(y_list)
        
        return {
            'x': x_arr, 'y': y_arr,
            'time': t,
            'max_height': max_height,
            'range': x_arr[-1] if len(x_arr) > 0 else 0
        }

print("✅ 物理エンジンの定義完了！")

## 3️⃣ インタラクティブシミュレーターの実行

### スライダーの説明
| パラメーター | 説明 | 単位 | グラフ内表記 |
|------------|------|------|-------------|
| **初速度** | ボールを投げる速さ | m/s | Initial Velocity |
| **発射角度** | 水平面からの角度 | 度 | Launch Angle |
| **初期高さ** | 発射点の高さ | m | Initial Height |
| **質量** | ボールの重さ | kg | Mass |
| **半径** | ボールの大きさ | m | Radius |
| **抗力係数** | 空気抵抗の強さ（球体≈0.47） | - | Drag Coefficient |

### グラフの凡例
- **青線 (Blue)**: No Air Resistance = 空気抵抗なし
- **赤線 (Red)**: With Air Resistance = 空気抵抗あり
- **緑点 (Green)**: Launch Point = 発射点

---

**🎯 下のスライダーを動かして実験してみましょう！**

In [ ]:
def plot_trajectories(v0, angle, height, mass, radius, drag_coef,
                       show_no_air, show_with_air):
    """
    軌跡をプロットする関数
    グラフのラベル・タイトル・凡例は英語で表示（文字化け防止）
    """
    fig, ax = plt.subplots(figsize=(12, 7))
    
    results = []
    
    # 空気抵抗なしの軌跡 (No Air Resistance)
    if show_no_air:
        data_no_air = PhysicsEngine.no_air_resistance(v0, angle, height)
        ax.plot(data_no_air['x'], data_no_air['y'], 'b-', linewidth=2.5,
                label=f'No Air Resistance (Range: {data_no_air["range"]:.1f}m)')
        results.append(('空気抵抗なし', data_no_air))
    
    # 空気抵抗ありの軌跡 (With Air Resistance)
    if show_with_air:
        data_with_air = PhysicsEngine.with_air_resistance(
            v0, angle, mass, radius, drag_coef, height
        )
        ax.plot(data_with_air['x'], data_with_air['y'], 'r-', linewidth=2.5,
                label=f'With Air Resistance (Range: {data_with_air["range"]:.1f}m)')
        results.append(('空気抵抗あり', data_with_air))
    
    # 発射点 (Launch Point)
    ax.plot(0, height, 'go', markersize=12, label='Launch Point', zorder=5)
    
    # 初速度ベクトル (Initial Velocity Vector)
    arrow_scale = 0.3
    dx = v0 * np.cos(np.radians(angle)) * arrow_scale
    dy = v0 * np.sin(np.radians(angle)) * arrow_scale
    ax.annotate('', xy=(dx, height + dy), xytext=(0, height),
                arrowprops=dict(arrowstyle='->', color='green', lw=2.5))
    
    # グラフ設定（英語表記）
    ax.set_xlabel('Distance [m]', fontsize=14)
    ax.set_ylabel('Height [m]', fontsize=14)
    ax.set_title(f'Projectile Motion Simulation\n'
                 f'v0={v0:.1f} m/s, Angle={angle:.1f} deg, h0={height:.1f} m',
                 fontsize=16)
    ax.legend(loc='upper right', fontsize=12)
    ax.grid(True, linestyle='--', alpha=0.7)
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0)
    ax.set_aspect('equal', adjustable='datalim')
    
    # 地面 (Ground)
    xlim = ax.get_xlim()
    ax.axhline(y=0, color='saddlebrown', linewidth=3)
    ax.fill_between([0, xlim[1]*1.1], -1, 0, color='forestgreen', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 結果表示（日本語で詳細表示）
    print("\n" + "="*60)
    print("📊 シミュレーション結果 (Simulation Results)")
    print("="*60)
    print(f"\n【入力パラメーター (Input Parameters)】")
    print(f"  初速度 (Initial Velocity): {v0:.1f} m/s")
    print(f"  発射角度 (Launch Angle): {angle:.1f}°")
    print(f"  初期高さ (Initial Height): {height:.1f} m")
    print(f"  質量 (Mass): {mass:.3f} kg")
    print(f"  半径 (Radius): {radius:.4f} m")
    print(f"  抗力係数 (Drag Coefficient): {drag_coef:.2f}")
    
    for name, data in results:
        print(f"\n【{name}】")
        print(f"  飛行時間 (Flight Time): {data['time']:.2f} s")
        print(f"  最大高さ (Max Height): {data['max_height']:.2f} m")
        print(f"  飛距離 (Range): {data['range']:.2f} m")
    
    # 差分表示
    if len(results) == 2:
        r1, r2 = results[0][1]['range'], results[1][1]['range']
        diff = r1 - r2
        pct = (diff / r1 * 100) if r1 > 0 else 0
        print(f"\n【差分 (Difference)】")
        print(f"  空気抵抗による飛距離減少: {diff:.2f} m ({pct:.1f}%)")
        print(f"  (Range reduction due to air resistance)")
    print("="*60)


# ウィジェットのスタイル設定
style = {'description_width': '120px'}
layout = Layout(width='400px')

# インタラクティブウィジェットの作成
interactive_plot = interactive(
    plot_trajectories,
    v0=widgets.FloatSlider(
        value=30, min=1, max=100, step=1,
        description='初速度 [m/s]:', style=style, layout=layout,
        continuous_update=False
    ),
    angle=widgets.FloatSlider(
        value=45, min=0, max=90, step=1,
        description='発射角度 [deg]:', style=style, layout=layout,
        continuous_update=False
    ),
    height=widgets.FloatSlider(
        value=0, min=0, max=50, step=1,
        description='初期高さ [m]:', style=style, layout=layout,
        continuous_update=False
    ),
    mass=widgets.FloatSlider(
        value=0.145, min=0.01, max=1.0, step=0.01,
        description='質量 [kg]:', style=style, layout=layout,
        continuous_update=False
    ),
    radius=widgets.FloatSlider(
        value=0.0366, min=0.01, max=0.2, step=0.005,
        description='半径 [m]:', style=style, layout=layout,
        continuous_update=False
    ),
    drag_coef=widgets.FloatSlider(
        value=0.47, min=0.1, max=1.5, step=0.01,
        description='抗力係数 Cd:', style=style, layout=layout,
        continuous_update=False
    ),
    show_no_air=widgets.Checkbox(
        value=True, description='空気抵抗なし (No Drag)',
        style=style
    ),
    show_with_air=widgets.Checkbox(
        value=True, description='空気抵抗あり (With Drag)',
        style=style
    )
)

display(interactive_plot)

## 4️⃣ プリセットボールで実験

いくつかの代表的なボールでシミュレーションを行います。

### グラフの凡例 (Legend)
- 各色の実線: 各ボールの軌跡（空気抵抗あり）
- 黒の破線 (Black dashed): Ideal (No Drag) = 理想軌跡（空気抵抗なし）

In [ ]:
# プリセットボールの定義
# 各ボールの物理パラメーター（質量、半径、抗力係数）
PRESET_BALLS = {
    'Baseball': {'mass': 0.145, 'radius': 0.0366, 'drag': 0.47, 'jp': '野球ボール'},
    'Soccer': {'mass': 0.43, 'radius': 0.11, 'drag': 0.25, 'jp': 'サッカーボール'},
    'Golf': {'mass': 0.046, 'radius': 0.0214, 'drag': 0.25, 'jp': 'ゴルフボール'},
    'Tennis': {'mass': 0.058, 'radius': 0.0335, 'drag': 0.55, 'jp': 'テニスボール'},
    'Basketball': {'mass': 0.625, 'radius': 0.12, 'drag': 0.47, 'jp': 'バスケットボール'},
    'Shot Put': {'mass': 7.26, 'radius': 0.06, 'drag': 0.47, 'jp': '砲丸'},
}

def compare_balls(v0=30, angle=45):
    """
    複数のボールの軌跡を比較
    """
    fig, ax = plt.subplots(figsize=(14, 8))
    colors = plt.cm.tab10(np.linspace(0, 1, len(PRESET_BALLS)))
    
    # 日本語で結果表示
    print(f"\n🎯 各ボールの飛距離比較")
    print(f"   条件: 初速度 {v0} m/s, 発射角度 {angle}°\n")
    print(f"{'ボール':<20} {'質量[kg]':>10} {'半径[m]':>10} {'飛距離[m]':>12} {'減少率':>10}")
    print("-" * 65)
    
    for i, (name, props) in enumerate(PRESET_BALLS.items()):
        # 空気抵抗なし（理想）
        ideal = PhysicsEngine.no_air_resistance(v0, angle)
        
        # 空気抵抗あり（現実）
        real = PhysicsEngine.with_air_resistance(
            v0, angle, props['mass'], props['radius'], props['drag']
        )
        
        # グラフに英語名でプロット
        ax.plot(real['x'], real['y'], '-', linewidth=2.5, color=colors[i],
                label=f"{name} ({real['range']:.1f}m)")
        
        # 結果を日本語で表示
        reduction = (1 - real['range'] / ideal['range']) * 100 if ideal['range'] > 0 else 0
        print(f"{props['jp']:<18} {props['mass']:>10.3f} {props['radius']:>10.4f} "
              f"{real['range']:>12.2f} {reduction:>9.1f}%")
    
    # 理想軌跡（空気抵抗なし）
    ideal = PhysicsEngine.no_air_resistance(v0, angle)
    ax.plot(ideal['x'], ideal['y'], 'k--', linewidth=2, alpha=0.7,
            label=f'Ideal (No Drag) ({ideal["range"]:.1f}m)')
    
    # グラフ設定（英語表記）
    ax.set_xlabel('Distance [m]', fontsize=14)
    ax.set_ylabel('Height [m]', fontsize=14)
    ax.set_title(f'Trajectory Comparison of Various Balls\n'
                 f'v0={v0} m/s, Angle={angle} deg', fontsize=16)
    ax.legend(loc='upper right', fontsize=11)
    ax.grid(True, linestyle='--', alpha=0.7)
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0)
    ax.set_aspect('equal', adjustable='datalim')
    
    plt.tight_layout()
    plt.show()
    
    print("\n※ グラフの凡例は英語、詳細結果は日本語で表示しています")

# ボール比較のインタラクティブウィジェット
interactive_balls = interactive(
    compare_balls,
    v0=widgets.IntSlider(value=30, min=10, max=80, step=5,
                         description='初速度 [m/s]:', style=style, layout=layout),
    angle=widgets.IntSlider(value=45, min=10, max=80, step=5,
                            description='発射角度 [deg]:', style=style, layout=layout)
)

display(interactive_balls)

## 5️⃣ 最適発射角度を探す

空気抵抗がある場合、最大飛距離を得るための最適角度は45°ではありません。
実際に確認してみましょう。

### グラフの見方
- **横軸 (X-axis)**: Launch Angle [deg] = 発射角度 [度]
- **縦軸 (Y-axis)**: Range [m] = 飛距離 [m]
- **青線 (Blue)**: No Air Resistance = 空気抵抗なし（最適角度: Optimal Angle）
- **赤線 (Red)**: With Air Resistance = 空気抵抗あり（最適角度: Optimal Angle）
- **灰色破線 (Gray dashed)**: 45 deg line = 45度ライン

In [ ]:
def find_optimal_angle(v0=30, mass=0.145, radius=0.0366, drag_coef=0.47):
    """
    最適発射角度を探索
    """
    angles = np.arange(5, 86, 1)
    ranges_no_air = []
    ranges_with_air = []
    
    for angle in angles:
        # 空気抵抗なし
        data_no = PhysicsEngine.no_air_resistance(v0, angle)
        ranges_no_air.append(data_no['range'])
        
        # 空気抵抗あり
        data_with = PhysicsEngine.with_air_resistance(
            v0, angle, mass, radius, drag_coef
        )
        ranges_with_air.append(data_with['range'])
    
    # 最適角度を見つける
    opt_idx_no = np.argmax(ranges_no_air)
    opt_idx_with = np.argmax(ranges_with_air)
    
    # プロット（英語表記）
    fig, ax = plt.subplots(figsize=(12, 6))
    
    ax.plot(angles, ranges_no_air, 'b-', linewidth=2.5,
            label=f'No Air Resistance (Optimal: {angles[opt_idx_no]} deg)')
    ax.plot(angles, ranges_with_air, 'r-', linewidth=2.5,
            label=f'With Air Resistance (Optimal: {angles[opt_idx_with]} deg)')
    
    # 最適点をマーク
    ax.plot(angles[opt_idx_no], ranges_no_air[opt_idx_no], 'bo', markersize=12)
    ax.plot(angles[opt_idx_with], ranges_with_air[opt_idx_with], 'ro', markersize=12)
    
    ax.axvline(x=45, color='gray', linestyle='--', alpha=0.5, label='45 deg line')
    
    # グラフ設定（英語表記）
    ax.set_xlabel('Launch Angle [deg]', fontsize=14)
    ax.set_ylabel('Range [m]', fontsize=14)
    ax.set_title(f'Launch Angle vs Range\n'
                 f'v0={v0} m/s, mass={mass} kg, radius={radius} m',
                 fontsize=16)
    ax.legend(loc='upper right', fontsize=12)
    ax.grid(True, linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    plt.show()
    
    # 結果を日本語で表示
    print(f"\n📊 結果 (Results):")
    print(f"  【空気抵抗なし (No Air Resistance)】")
    print(f"    最適角度 (Optimal Angle): {angles[opt_idx_no]}°")
    print(f"    最大飛距離 (Max Range): {ranges_no_air[opt_idx_no]:.2f} m")
    print(f"\n  【空気抵抗あり (With Air Resistance)】")
    print(f"    最適角度 (Optimal Angle): {angles[opt_idx_with]}°")
    print(f"    最大飛距離 (Max Range): {ranges_with_air[opt_idx_with]:.2f} m")
    print(f"\n  角度の差: {angles[opt_idx_no] - angles[opt_idx_with]}°")
    print(f"  → 空気抵抗があると、最適角度は45°より小さくなる！")

# インタラクティブウィジェット
interactive_optimal = interactive(
    find_optimal_angle,
    v0=widgets.IntSlider(value=30, min=10, max=80, step=5,
                         description='初速度 [m/s]:', style=style, layout=layout),
    mass=widgets.FloatSlider(value=0.145, min=0.01, max=1.0, step=0.01,
                             description='質量 [kg]:', style=style, layout=layout),
    radius=widgets.FloatSlider(value=0.0366, min=0.01, max=0.15, step=0.005,
                               description='半径 [m]:', style=style, layout=layout),
    drag_coef=widgets.FloatSlider(value=0.47, min=0.1, max=1.0, step=0.05,
                                  description='抗力係数 Cd:', style=style, layout=layout)
)

display(interactive_optimal)

## 6️⃣ アニメーション表示

ボールの飛行をアニメーションで確認します。

### アニメーションの凡例
- **青 (Blue)**: No Air Resistance = 空気抵抗なし
- **赤 (Red)**: With Air Resistance = 空気抵抗あり

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def create_animation(v0=30, angle=45, mass=0.145, radius=0.0366, drag_coef=0.47):
    """
    放物運動のアニメーションを作成
    """
    # 軌跡データの計算
    data_no_air = PhysicsEngine.no_air_resistance(v0, angle)
    data_with_air = PhysicsEngine.with_air_resistance(v0, angle, mass, radius, drag_coef)
    
    # フレーム数の調整（両方の軌跡を同時に終わらせる）
    n_frames = 100
    
    # インデックスの補間
    idx_no = np.linspace(0, len(data_no_air['x'])-1, n_frames).astype(int)
    idx_with = np.linspace(0, len(data_with_air['x'])-1, n_frames).astype(int)
    
    # Figure作成
    fig, ax = plt.subplots(figsize=(12, 7))
    
    # 軸設定
    max_x = max(data_no_air['x'].max(), data_with_air['x'].max()) * 1.1
    max_y = max(data_no_air['y'].max(), data_with_air['y'].max()) * 1.2
    ax.set_xlim(0, max_x)
    ax.set_ylim(0, max_y)
    
    # グラフ設定（英語表記）
    ax.set_xlabel('Distance [m]', fontsize=14)
    ax.set_ylabel('Height [m]', fontsize=14)
    ax.set_title(f'Projectile Motion Animation\n'
                 f'v0={v0} m/s, Angle={angle} deg', fontsize=16)
    ax.grid(True, linestyle='--', alpha=0.7)
    ax.set_aspect('equal', adjustable='box')
    
    # 地面 (Ground)
    ax.axhline(y=0, color='saddlebrown', linewidth=3)
    
    # 軌跡ライン（最初は空）
    line_no, = ax.plot([], [], 'b-', linewidth=2, alpha=0.5, label='No Air Resistance')
    line_with, = ax.plot([], [], 'r-', linewidth=2, alpha=0.5, label='With Air Resistance')
    
    # ボール
    ball_no, = ax.plot([], [], 'bo', markersize=15)
    ball_with, = ax.plot([], [], 'ro', markersize=15)
    
    ax.legend(loc='upper right', fontsize=12)
    
    def init():
        line_no.set_data([], [])
        line_with.set_data([], [])
        ball_no.set_data([], [])
        ball_with.set_data([], [])
        return line_no, line_with, ball_no, ball_with
    
    def animate(frame):
        # 軌跡の更新
        i_no = idx_no[frame]
        i_with = idx_with[frame]
        
        line_no.set_data(data_no_air['x'][:i_no+1], data_no_air['y'][:i_no+1])
        line_with.set_data(data_with_air['x'][:i_with+1], data_with_air['y'][:i_with+1])
        
        ball_no.set_data([data_no_air['x'][i_no]], [data_no_air['y'][i_no]])
        ball_with.set_data([data_with_air['x'][i_with]], [data_with_air['y'][i_with]])
        
        return line_no, line_with, ball_no, ball_with
    
    anim = FuncAnimation(fig, animate, init_func=init,
                         frames=n_frames, interval=50, blit=True)
    
    plt.close(fig)
    return HTML(anim.to_jshtml())

# アニメーション表示
print("🎬 アニメーションを生成中...（数秒かかります）")
print("   Generating animation... (takes a few seconds)\n")
anim_html = create_animation(v0=40, angle=50)
display(anim_html)

## 7️⃣ 学習チェックポイント

以下の実験を試して、理解を深めましょう：

### 実験1: 空気抵抗の影響
- [ ] 空気抵抗あり/なしを比較して、軌跡の違いを確認
- [ ] 速度を上げると空気抵抗の影響はどうなる？

### 実験2: 最適角度
- [ ] 空気抵抗なしでの最適角度（45°）を確認
- [ ] 空気抵抗ありでは何度が最適？

### 実験3: ボールの性質
- [ ] 重いボール vs 軽いボール → どちらが遠くへ飛ぶ？
- [ ] 大きいボール vs 小さいボール → どちらが遠くへ飛ぶ？

### 実験4: 理論値との比較
- [ ] 空気抵抗なしでの飛距離 R = v₀²sin(2θ)/g を計算して確認

In [ ]:
# 理論値の確認
def verify_theory(v0=30, angle=45):
    """
    理論値とシミュレーション値を比較
    Theory vs Simulation comparison
    """
    g = 9.81
    angle_rad = np.radians(angle)
    
    # 理論値 (Theoretical values)
    R_theory = v0**2 * np.sin(2 * angle_rad) / g
    H_theory = v0**2 * np.sin(angle_rad)**2 / (2 * g)
    T_theory = 2 * v0 * np.sin(angle_rad) / g
    
    # シミュレーション値 (Simulation values)
    data = PhysicsEngine.no_air_resistance(v0, angle)
    
    print("📐 理論値 vs シミュレーション値（空気抵抗なし）")
    print("   Theory vs Simulation (No Air Resistance)")
    print("="*60)
    print(f"条件: 初速度 v0={v0} m/s, 発射角度 θ={angle}°")
    print()
    print(f"{'項目':<20} {'理論値':>12} {'シミュレーション':>15} {'誤差':>10}")
    print(f"{'(Parameter)':<20} {'(Theory)':>12} {'(Simulation)':>15} {'(Error)':>10}")
    print("-"*60)
    print(f"{'飛距離 (Range) [m]':<20} {R_theory:>12.4f} {data['range']:>15.4f} {abs(R_theory-data['range']):>10.6f}")
    print(f"{'最大高さ (Height) [m]':<20} {H_theory:>12.4f} {data['max_height']:>15.4f} {abs(H_theory-data['max_height']):>10.6f}")
    print(f"{'飛行時間 (Time) [s]':<20} {T_theory:>12.4f} {data['time']:>15.4f} {abs(T_theory-data['time']):>10.6f}")
    print()
    print("✅ 理論値とシミュレーション値がほぼ一致していることを確認！")
    print("   (Theory and simulation values match well!)")

verify_theory(30, 45)

---

## 📖 まとめ (Summary)

このシミュレーターで学んだこと：

1. **空気抵抗なしの放物運動**は解析的に解ける（二次関数）
2. **空気抵抗ありの場合**は数値計算（Runge-Kutta法）が必要
3. 空気抵抗があると **最適発射角度は45°より小さくなる**
4. **重いボール**ほど空気抵抗の影響を受けにくい
5. **速度の2乗**に比例して空気抵抗は増加する

---

### グラフ用語の対訳表 (Graph Terminology)

| English (in graphs) | 日本語 |
|---------------------|--------|
| Distance | 距離 |
| Height | 高さ |
| Range | 飛距離 |
| Launch Angle | 発射角度 |
| No Air Resistance | 空気抵抗なし |
| With Air Resistance | 空気抵抗あり |
| Optimal Angle | 最適角度 |
| Launch Point | 発射点 |
| Ideal (No Drag) | 理想軌跡（抵抗なし） |

---

*Created for physics education* 🎓